In [ ]:
import pandas as pd
import altair as alt
alt.data_transformers.enable('default', max_rows=None) 
alt.renderers.enable('notebook')

In [ ]:
df = pd.read_csv('game_sales.csv')

1. amount of sales by platform

2. amount of sales by platform by region

In [ ]:
alt.Chart(df).mark_bar().encode(
    x=alt.X(
        'Platform',
        sort=alt.EncodingSortField(
            field="Global_Sales",  # The field to use for the sort
            op="sum",  # The operation to run on the field prior to sorting
            order="descending"  # The order to sort in
        )),
    y=alt.Y('sum(Global_Sales)'),
    tooltip=alt.Tooltip([
        'sum(Global_Sales)'
    ]),

).transform_filter("datum.Global_Sales>2").properties(width=800,
                                                            height=600)

amount of sales by platform by year with selection

In [ ]:
platform_selection = alt.selection_single(fields=['Platform'])
color=alt.condition(platform_selection, alt.value('darkblue'), alt.value('lightgray'))

barchart = alt.Chart(df).mark_bar().encode(
    y=alt.Y(
        'Platform',
        sort=alt.EncodingSortField(
            field="Global_Sales",  # The field to use for the sort
            op="sum",  # The operation to run on the field prior to sorting
            order="descending"  # The order to sort in
        )),
    x=alt.X('sum(Global_Sales)'),
    tooltip=alt.Tooltip(['sum(Global_Sales)']),
    color=color
).transform_filter("datum.Global_Sales>5").properties(
    width=300, height=400).add_selection(platform_selection)

year_histogram=alt.Chart(df).mark_bar().encode(
    x=alt.X('Year', scale=alt.Scale(domain=(1970, 2020))),
    y=alt.Y('sum(Global_Sales)', scale=alt.Scale(domain=(0, 600))),
    tooltip=alt.Tooltip(['sum(Global_Sales)']),
    color=color
).properties(width=500, height=400)

base_year_hist = alt.Chart(df).mark_bar().encode(
    x=alt.X('Year', scale=alt.Scale(domain=(1980, 2020))),
    y=alt.Y('sum(Global_Sales)', scale=alt.Scale(domain=(0, 600))),
    tooltip=alt.Tooltip(['sum(Global_Sales)']),
).properties(width=500, height=400)

background = base_year_hist.encode(
    color=alt.value('#ddd')
)
interactive_year_hist = base_year_hist.transform_filter(platform_selection)


layered = alt.layer(
    background,
    interactive_year_hist)
alt.hconcat(layered, barchart)

critics score by platform

In [ ]:
# alt.Chart(df).mark_circle(opacity=0.5, size = 50).encode(
#     x = alt.X('Critic_Score'),
#     y = alt.Y('Platform', sort = alt.Sort(field = 'Critic_Score', op = 'mean', order = 'descending')),
#     color = alt.Color('Genre'),
#     tooltip = alt.Tooltip(['Critic_Score'])
# ).transform_filter("datum.Global_Sales>5").properties(width = 850)

most 

Critic_Score VS User_Score for each genre

In [ ]:
base_line = alt.Chart(df).mark_line().encode(
    x=alt.X('User_Score', aggregate='mean', scale=alt.Scale(zero=False)),
    x2=alt.X2('Critic_Score', aggregate='mean'),
    y=alt.Y('Genre',
            sort=alt.Sort(field='Critic_Score', op='mean',
                          order='descending')),
).transform_filter('datum.User_Score>0')

critics = base_line.mark_point(size=20).encode(
    x=alt.X(
        'Critic_Score',
        aggregate='mean',
    ),
    y=alt.Y('Genre',
            sort=alt.Sort(field='Critic_Score', op='mean',
                          order='descending')),
    color=alt.value('red'))
user = base_line.mark_point(size=20).encode(x=alt.X(
    'User_Score',
    aggregate='mean',
),
                                            y=alt.Y('Genre',
                                                    sort=alt.Sort(
                                                        field='Critic_Score',
                                                        op='mean',
                                                        order='descending')),
                                            color=alt.value('blue'))
layered = alt.layer(base_line, critics, user)

legend_data = alt.Data(values=[{
    "label": "Users Review",
    "color": "blue"
}, {
    "label": "Critic Review",
    "color": "red"
}])
legend_graph = alt.Chart(legend_data).mark_point().encode(
    y=alt.Y('label:O', title = ''),
    color=alt.condition(alt.datum.label == "Users Review", alt.value('blue'), alt.value('red'))
)
legend_graph
alt.hconcat(layered, legend_graph)